# QPSO

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import numpy as np
import torch
from typing import List, Tuple

from Helper_invert_1d import initialize_inversion, fitness_s, unpack_qpso_solutions


def invert_first_output_qpso_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model_path: str,
    x_fixed_raw: np.ndarray,        # fixed inputs, raw scale (shape depends on model)
    actual_x_adj_raw: np.ndarray,   # actual adjustable inputs, raw scale (4,)
    target_raw: float,              # desired raw value of first output
    pop_size: int,
    num_iterations: int,
    beta: float,
    hidden_dims: List[int],
    activation: str,
    x_adj_intervals_raw: np.ndarray,  # shape (4,2), raw [min,max] per adjustable input
    Q_bounds: Tuple[float, float],     # (Q_min, Q_max) in raw space
    k: int                             # number of top solutions to return (excluding actual)
) -> List[Tuple[np.ndarray, np.ndarray, float]]:
    """
    Invert the first network output under the constraint x1 = Q * x0
    using Quantum-behaved PSO (QPSO). Returns up to k+1 solutions:
      (adjustable_inputs_raw (4,), all_outputs_raw (6,), residual_raw).
    The first element is the actual input/output pair.

    Steps:
      1. Initialize scalers, model, and compute the “actual” prediction.
      2. Build scaled search bounds and target for fitness.
      3. Run QPSO to find candidate scaled inputs [x0_s, x2_s, x3_s, Q].
      4. Select top-k unique candidates.
      5. Inverse-scale candidates to raw inputs, predict full outputs, compute residuals.
    """
    # [1] Initialization (scalers, model, scaling parameters, actual result, bounds, target)
    init = initialize_inversion(
        x_scaler_path, y_scaler_path, model_path,
        hidden_dims, activation,
        x_fixed_raw, actual_x_adj_raw,
        target_raw, x_adj_intervals_raw, Q_bounds
    )
    model         = init["model"]
    x_fixed_s     = init["x_fixed_s"]
    means_x       = init["means_x"]
    scales_x      = init["scales_x"]
    y_scaler      = init["y_scaler"]
    results       = init["results"]
    min_vec       = init["min_vec"]
    max_vec       = init["max_vec"]
    target_scaled = init["target_scaled"]

    # [2] Initialize QPSO population and personal/global bests
    population = [np.random.uniform(min_vec, max_vec) for _ in range(pop_size)]
    pbest      = population.copy()
    pbest_f    = [
        fitness_s(cand, model, x_fixed_s, means_x, scales_x, target_scaled)
        for cand in population
    ]
    gbest = pbest[int(np.argmin(pbest_f))]

    # [3] QPSO main loop
    for _ in range(num_iterations):
        mbest = np.mean(pbest, axis=0)
        for i in range(pop_size):
            phi  = np.random.rand(4)
            P_i  = phi * pbest[i] + (1 - phi) * gbest
            u    = np.random.rand(4)
            sign = np.where(np.random.rand(4) < 0.5, 1.0, -1.0)
            x_i  = P_i + sign * beta * np.abs(mbest - population[i]) * np.log(1.0 / u)
            # clamp into bounds
            x_i  = np.minimum(np.maximum(x_i, min_vec), max_vec)

            population[i] = x_i
            f_val = fitness_s(x_i, model, x_fixed_s, means_x, scales_x, target_scaled)
            if f_val < pbest_f[i]:
                pbest[i], pbest_f[i] = x_i.copy(), f_val

        gbest = pbest[int(np.argmin(pbest_f))]

    # [4] Select top-k unique scaled candidates
    sorted_candidates = sorted(zip(pbest_f, pbest), key=lambda t: t[0])
    unique_candidates, seen = [], set()
    for _, cand in sorted_candidates:
        key = tuple(np.round(cand, 6))
        if key in seen:
            continue
        seen.add(key)
        unique_candidates.append(cand)
        if len(unique_candidates) >= k:
            break

    # [5] Inverse-scale and compute raw outputs + residuals
    other_results = unpack_qpso_solutions(
        unique_candidates,
        means_x,
        scales_x,
        x_fixed_s,
        model,
        y_scaler,
        target_raw
    )
    results.extend(other_results)

    return results
